# 辞書作成 ＋ yaml×学習データ 整合チェック（パイプライン①）

**全体パイプライン**: **①yaml＋学習データ→辞書作成（このノート）** → ②A/B学習(verify07) → ③学習モデル＋年齢論理でベクトル生成(verify08b) → ④トリアージ

このノートの役割:
1. aligned-yaml（choicesに `code_label`＝データ数値）＋ node_map ＋ 学習データ(07_21) から
   `dictionary/node_questions_A.json`（質問のみ）/ `node_questions_B.json`（質問＋選択肢）を作る。
2. **yaml の code_label とデータの label(code) のズレ**を全ノードで点検し、
   「ズレあり／どのノード／種類（yaml不足・yaml余分）／どのプロトコルで多いか」をレポートする。

> ズレが残っている間は辞書を正典にしない。**yamlを直してズレ0にしてからcommit → verify07学習へ。**


In [ ]:
# ===== 設定 =====
import os, glob, re, csv, json, io
import yaml as _yaml
from collections import Counter, defaultdict

DATA_DIR = '07_21'                                                            # 学習データ(症状別CSV)フォルダ
YAML     = 'transition_diagram/protocol_spreadsheet_aligned202607211237.yaml' # aligned-yaml(最新)
NODE_MAP = 'transition_diagram/node_map.json'
OUT_DIR  = 'dictionary'
os.makedirs(OUT_DIR, exist_ok=True)
print('DATA_DIR =', DATA_DIR); print('YAML     =', YAML)

In [ ]:
# ===== 読み込み: yaml / node_map / 学習データ =====
doc = _yaml.safe_load(open(YAML, encoding='utf-8'))
def _nodes(d):
    for s in ('entry_flow', 'common_vitals'):
        for n in d.get(s, []):
            yield n
    for p in d.get('protocols', []):
        for n in p.get('nodes', []):
            yield n

yq, ylab = {}, {}                       # yaml_id -> 質問 / {code_label:text}
for n in _nodes(doc):
    if 'id' not in n:
        continue
    if 'question' in n:
        yq[n['id']] = re.sub(r'^\s*[（(].*?[）)]\s*', '', str(n['question'])).strip()
    if 'choices' in n:
        d = {}
        for c in n['choices']:
            try:
                d[int(c.get('code_label'))] = str(c.get('text', ''))   # code_label=None等はここで弾かれ『不足』検出
            except (TypeError, ValueError):
                pass
        ylab[n['id']] = d

nm = json.load(open(NODE_MAP, encoding='utf-8'))                       # yaml_id -> bert_node
bq = {b: yq[y]   for y, b in nm.items() if y in yq}                    # bert_node -> 質問
bl = {b: ylab[y] for y, b in nm.items() if y in ylab}                  # bert_node -> {code_label:text}
cq = {y: yq[y]   for y in yq   if y.startswith('common_')}             # common_* は素名でも拾う
cl = {y: ylab[y] for y in ylab if y.startswith('common_')}

def _read(p):
    for e in ('utf-8-sig', 'cp932'):
        try:
            return list(csv.DictReader(io.StringIO(open(p, 'rb').read().decode(e))))
        except UnicodeDecodeError:
            continue
    return []
def _on(v): return str(v).strip().lower() in ('true', '1', '1.0')
def _code(v):
    s = str(v).strip()
    if s in ('', 'nan', 'None'):
        return None
    try:
        return int(float(s))
    except ValueError:
        return None

# 同じプロトコル番号のCSVが複数あれば最新タイムスタンプの1つだけ使う（旧データ混入防止）
def _file_ts(p):
    _cs = re.findall(r'\d{8,14}', os.path.basename(p))
    if _cs:
        return max(int(c.ljust(14, '0')) for c in _cs)
    return int(os.path.getmtime(p))
_paths = {}
for p in sorted(glob.glob(os.path.join(DATA_DIR, '*.csv'))):
    _pid = os.path.basename(p)[:2]
    if _pid not in _paths or _file_ts(p) > _file_ts(_paths[_pid]):
        _paths[_pid] = p
_dropped = sorted(set(glob.glob(os.path.join(DATA_DIR, '*.csv'))) - set(_paths.values()))
if _dropped:
    print('[最新版のみ採用] 旧版を除外:', [os.path.basename(p) for p in _dropped])

data = defaultdict(Counter)             # bert_node -> {code:件数}
for p in sorted(_paths.values()):
    pid = os.path.basename(p)[:2]
    rows = _read(p)
    if not rows:
        continue
    for k in [c[3:] for c in rows[0] if c.startswith('is_')]:
        for r in rows:
            if _on(r.get('is_' + k, '')):
                c = _code(r.get('label_' + k, ''))
                if c is not None:
                    data[f'{pid}_{k}'][c] += 1

def q_of(node):   return bq.get(node) or cq.get(node[3:])
def lab_of(node): return bl.get(node) or cl.get(node[3:])
print('データノード数:', len(data), '/ yaml質問:', len(bq), '/ node_map:', len(nm))

In [ ]:
# ===== 辞書作成（A/B）＋整合ゲート付き保存 =====
# ズレ(データのcodeがyamlに無い/余分)があるノードを検出。ズレ0なら dictionary/ に保存(正典)、
# ズレ有りなら dictionary/_draft/ に退避し canonical を上書きしない（未整合辞書の誤commit防止）。
_bad = []
for node in sorted(data):
    dcodes = set(c for c in data[node] if c != 0)
    lab = lab_of(node)
    if lab is None or (dcodes - set(lab)) or (set(lab) - dcodes):
        _bad.append(node)

A_out, B_out = {}, {}
for node in sorted(data):
    q = q_of(node)
    lab = lab_of(node) or {}
    order = sorted(lab)
    choices = [lab[i] for i in order]
    suffix = ('（選択肢: ' + ' / '.join(['%d)%s' % (i, lab[i]) for i in order]) + '）') if lab else None
    A_out[node] = {'question': q}
    B_out[node] = {'question': q, 'choices': choices, 'n_choices': len(choices),
                   'max_data_code': (max(data[node]) if data[node] else 0),
                   'prompt_suffix': suffix, 'source': 'yaml_code_label'}

_dst = OUT_DIR if not _bad else os.path.join(OUT_DIR, '_draft')
os.makedirs(_dst, exist_ok=True)
json.dump({'_meta': {'purpose': 'A(質問のみ)', 'yaml': os.path.basename(YAML), 'n_nodes': len(A_out)}, 'questions': A_out},
          open(os.path.join(_dst, 'node_questions_A.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
json.dump({'_meta': {'purpose': 'B(質問＋選択肢)', 'yaml': os.path.basename(YAML), 'n_nodes': len(B_out)}, 'questions': B_out},
          open(os.path.join(_dst, 'node_questions_B.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
if _bad:
    print('⚠ ズレ%dノードあり → %s に保存（canonical dictionary/ は上書きしない）。' % (len(_bad), _dst))
    print('   yamlを直して再実行し、ズレ0になったら dictionary/ に出ます: %s' % _bad)
else:
    print('✅ ズレ0 → dictionary/ に保存（正典）。')

In [ ]:
# ===== yaml × 学習データ 整合レポート =====
import pandas as pd
from IPython.display import display
rows = []
for node in sorted(data):
    dcodes = set(c for c in data[node] if c != 0)          # データが使う非該当以外のcode
    lab = lab_of(node)
    if lab is None:
        rows.append({'node': node, 'proto': node[:2], 'data_code': sorted(dcodes),
                     'yaml_label': None, '状態': 'yaml無', '不足': sorted(dcodes), '余分': []})
        continue
    yl = set(lab)
    miss = sorted(dcodes - yl)                              # データにあるがyamlに無い→学習で選択肢文が出ない
    extra = sorted(yl - dcodes)                             # yamlにあるがデータが使わない番号
    st = 'OK' if (not miss and not extra) else ('ズレ:yaml不足' if miss else 'ズレ:yaml余分')
    rows.append({'node': node, 'proto': node[:2], 'data_code': sorted(dcodes),
                 'yaml_label': sorted(yl), '状態': st, '不足': miss, '余分': extra})
rep = pd.DataFrame(rows)
bad = rep[rep['状態'] != 'OK']
print('===== 整合サマリ =====  全%dノード / OK=%d / ズレ=%d' % (len(rep), (rep['状態'] == 'OK').sum(), len(bad)))
if len(bad):
    print('■ ズレが多いプロトコル:', dict(Counter(bad['proto']).most_common()))
    print('■ ズレ内訳:')
    for _, r in bad.iterrows():
        print('  [%s] %s  data=%s / yaml=%s  不足%s 余分%s'
              % (r['状態'], r['node'], r['data_code'], r['yaml_label'], r['不足'], r['余分']))
else:
    print('✅ 全ノードで yaml と学習データの label が一致。辞書は正典にできます。')
rep.to_csv(os.path.join(OUT_DIR, 'yaml_data_alignment_report.csv'), index=False, encoding='utf-8-sig')
print('レポート保存: dictionary/yaml_data_alignment_report.csv')
display(bad if len(bad) else rep.head())

In [ ]:
# ===== ズレノードの深掘り（yaml選択肢 と データ各codeの実応答を並べて確認）=====
def _tail(p):
    ls = [x.strip() for x in re.split(r'[\n\r]+', str(p)) if x.strip()]
    return ls[-1][:60] if ls else ''
def _samples(node):
    pid, k = node[:2], node[3:]
    for p in glob.glob(os.path.join(DATA_DIR, pid + '_*.csv')):
        rows = _read(p)
        if rows and ('is_' + k) in rows[0]:
            out = {}
            for r in rows:
                if _on(r.get('is_' + k, '')):
                    c = _code(r.get('label_' + k, ''))
                    if c is not None and c not in out:
                        out[c] = _tail(r['ペア'])
            return out
    return {}

for node in (list(bad['node']) if len(bad) else []):
    lab = lab_of(node) or {}
    print('=' * 70)
    print(node)
    print('  yaml選択肢:', {i: lab[i] for i in sorted(lab)})
    print('  データ各codeの実応答例:')
    s = _samples(node)
    for c in sorted(s):
        print('    code%d: %s' % (c, s[c]))

## まとめ・使い方
- **このノートを回す** → `dictionary/node_questions_A.json` / `_B.json` を（yaml＋データから）再生成。
- **整合レポート**（`dictionary/yaml_data_alignment_report.csv`）で **yaml↔学習データのズレ** を確認:
  - `ズレ:yaml不足` … データにあるcodeの選択肢文がyamlに無い（`code_label`欠落等）→ **yaml修正が必要**。
  - `ズレ:yaml余分` … yamlにデータが使わない番号がある（末尾なら無害／途中挿入なら意味ズレ）→ 深掘りセルで実応答と照合。
- **ズレを直す（yaml側）→ 再実行 → ズレ0 → commit** → verify07(A/B)学習へ。
- 辞書は「**データのlabelに一致した選択肢文**」であることが前提（A/B学習・推論の整合に必須）。
